# Setup

## Import Statements

In [1]:
import importlib
import session_organizer
from sentence_transformers import SentenceTransformer
import pandas as pd
# Only needed if you want to reload the module after making changes
importlib.reload(session_organizer)

<module 'session_organizer' from 'c:\\Users\\jdv223\\OneDrive - University of Kentucky\\Programming\\AI\\Session Creation Package\\session_organizer.py'>

## Step 1: Load and Examine Data

In [2]:
# First, examine the Excel file structure
file_path = "1.29.25 Abstracts.xlsx"
df_temp = pd.read_excel(file_path)

print("Available columns:")
for i, col in enumerate(df_temp.columns):
    print(f"{i}: {col}")

print(f"\nFile contains {len(df_temp)} rows and {len(df_temp.columns)} columns")
print("\nFirst few rows preview:")
print(df_temp.head())

Available columns:
0: Session
1: Submission Name
2: Abstract-Character max 4000-Abstracts will only be used to evaluate quality of talk and topic. They will not be published or able to be edited later.
3: Submission ID - 7 digits
4: Technical Community
5: Profile: First Name
6: Profile: Last Name

File contains 1606 rows and 7 columns

First few rows preview:
                                             Session  \
0  Value-Added Chemicals Products and Materials t...   
1  Generative AI and Large Multimodal model for A...   
2  Generative AI and Large Multimodal model for A...   
3  Nutrient Removal, Recovery and Recycling: Manu...   
4    Erosion Control and Sediment Transport Research   

                                     Submission Name  \
0  Repurposing of low-value biomass into engineer...   
1  AI Tools and Text Embedding for Session Organi...   
2  Automatic ASABE AIM Session Creation using Mac...   
3  Soil and Wastewater Effluent Properties of Oil...   
4  MODELLING GULLIES 

In [3]:
# Define your column selections based on the output above
TITLE_COLUMN = 'Submission Name'  # Update based on your file
ABSTRACT_COLUMN = 'Abstract-Character max 4000-Abstracts will only be used to evaluate quality of talk and topic. They will not be published or able to be edited later.'  # Update based on your file  
ID_COLUMN = 'Submission ID - 7 digits'  # Update based on your file



## Step 2: Load Embedding Model

In [4]:
# Available embedding models
EMBEDDING_MODELS = {
    "all-MiniLM-L6-v2": "sentence-transformers/all-MiniLM-L6-v2",
    "all-mpnet-base-v2": "sentence-transformers/all-mpnet-base-v2", 
    "paraphrase-MiniLM-L6-v2": "sentence-transformers/paraphrase-MiniLM-L6-v2",
    "cde-small-v1": "jxm/cde-small-v1",
    "cde-small-v2": "jxm/cde-small-v2",
}

# Select model (change as needed)
selected_model = "all-MiniLM-L6-v2"
model_name = EMBEDDING_MODELS[selected_model]

print(f"Loading embedding model: {model_name}")
embedding_model = SentenceTransformer(model_name, trust_remote_code=True)

if hasattr(embedding_model, 'model_card_data') and embedding_model.model_card_data:
    base_model = getattr(embedding_model.model_card_data, 'base_model', 'Unknown')
    print(f"Base model: {base_model}")
else:
    print(f"Model loaded: {model_name}")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
Base model: sentence-transformers/all-MiniLM-L6-v2


# Process Steps

## No Hybrid Sessions Example

### Load Data

In [5]:
# Load the data using the session_organizer function
df, title_column, abstract_column, abstract_id_column, topic_column = session_organizer.load_presentations(
    file_path, 
    Title_name=TITLE_COLUMN,
    Abstract_name=ABSTRACT_COLUMN,
    Abstract_ID_name=ID_COLUMN
)

print(f"Loaded {len(df)} presentations successfully")
print(f"Title column: {title_column}")
print(f"Abstract column: {abstract_column}")
print(f"ID column: {abstract_id_column}")
print(f"Topic column: {topic_column}")

Loaded 1601 presentations successfully
Title column: Title
Abstract column: Abstract
ID column: Abstract ID
Topic column: Title and Abstract


### Perform the Embedding

In [6]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', trust_remote_code=True)
print(f"Base model: {embedding_model.model_card_data.base_model}")
df_presentation_embeddings = session_organizer.embed_documents(df, topic_column, embedding_model)
print(f"Embeddings shape: {df_presentation_embeddings.shape}")
print(f"Embedding model used: {df_presentation_embeddings[session_organizer.COLUMNS['EMBEDDING_MODEL']].iloc[0]}")

Base model: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/51 [00:00<?, ?it/s]

Embeddings shape: (1601, 385)
Embedding model used: Unknown (sentence-transformers/all-MiniLM-L6-v2)


### Remove Duplicates and Near-Duplicates

In [7]:
similarity_threshold = 0.99
# Remove near-duplicate presentations based on the similarity threshold
df, df_presentation_embeddings = session_organizer.remove_duplicates(df, df_presentation_embeddings, similarity_func=embedding_model.similarity, threshold=similarity_threshold)

Near duplicate found: Index 41 and Index 42 (Similarity: 1.0000).
Near duplicate found: Index 77 and Index 78 (Similarity: 1.0000).
Near duplicate found: Index 77 and Index 79 (Similarity: 1.0000).
Near duplicate found: Index 78 and Index 79 (Similarity: 1.0000).
Near duplicate found: Index 156 and Index 157 (Similarity: 1.0000).
Near duplicate found: Index 221 and Index 222 (Similarity: 1.0000).
Near duplicate found: Index 223 and Index 224 (Similarity: 0.9930).
Near duplicate found: Index 269 and Index 270 (Similarity: 1.0000).
Near duplicate found: Index 279 and Index 280 (Similarity: 1.0000).
Near duplicate found: Index 325 and Index 326 (Similarity: 1.0000).
Near duplicate found: Index 354 and Index 355 (Similarity: 1.0000).
Near duplicate found: Index 410 and Index 411 (Similarity: 1.0000).
Near duplicate found: Index 458 and Index 459 (Similarity: 0.9998).
Near duplicate found: Index 476 and Index 477 (Similarity: 1.0000).
Near duplicate found: Index 496 and Index 1348 (Similari

### Create Sessions

In [8]:
session_column_name = 'Session Code'
df_sessions, labels, metadata = session_organizer.create_sessions_w_hybrid(df, embedding_model.similarity, df_presentation_embeddings=df_presentation_embeddings,
                                                                                     max_sessions=100, min_session_size=8, tree_merge_stop=1, cluster_column_name=session_column_name)
df[session_column_name] = labels
print(f"Created {metadata['n_clusters']} sessions with {metadata['n_assigned_items']} presentations.")
print(f"Unassigned Presentations: {metadata['n_unassigned_items']}")

Created 100 sessions with 1559 presentations.
Unassigned Presentations: 0


### Analyze Sessions

- session_coherence = "Are presentations within this session similar?" (internal session quality)
- session_distinctiveness = "Is this session's topic unique compared to others?" (relative session positioning)
- presentation_session_fit = "Does this presentation match the topic of others in the session?" (presentation fit)

Session Coherence measures cluster cohesion. It reflects how tighly grouped the topic of presentations within the session are.

Session Distinctiveness measures how unique each session's topic is. High values mean the session has a clear, focused theme that's different from other sessions. Low values suggest either the session mixes different topics or overlaps too much with other sessions.

Presentation-Session Fit is an individual presentations's average similarity to other presentation in its session. Generically, it can be referred to as "within_cluster_fit", "cluster_membership_strength", or "local_cohesion_score".

In [9]:
embeddings_only = df_presentation_embeddings.drop(columns=[session_organizer.COLUMNS['EMBEDDING_MODEL']])
pres_similarities_matrix = embedding_model.similarity(embeddings_only.values, embeddings_only.values)
# Convert to numpy if needed
if hasattr(pres_similarities_matrix, 'cpu'):
    pres_similarities_matrix = pres_similarities_matrix.cpu().numpy()
elif hasattr(pres_similarities_matrix, 'numpy'):
    pres_similarities_matrix = pres_similarities_matrix.numpy()

df['presentation_session_fit'],df_sessions['session_coherence'], df_sessions['session_distinctiveness'], df_session_session_similarity  = session_organizer.calculate_placement_metrics(
    df_presentations=df,
    df_sessions=df_sessions,
    pres_similarities_matrix=pres_similarities_matrix,
    session_column_name=session_column_name
)

### Create Session Titles & Keywords

#### Ollama

In [10]:
# Test if Ollama is accessible
import requests
try:
    response = requests.get("http://localhost:11434/api/tags")
    print(f"Ollama status: {response.status_code}")
    if response.status_code == 200:
        models = response.json()['models']
        print(f"Available models: {[m['name'] for m in models]}")
    else:
        print("Ollama server not responding correctly")
except Exception as e:
    print(f"Cannot connect to Ollama: {e}")
    print("Make sure to run 'ollama serve' first")

Ollama status: 200
Available models: ['llama3.2:latest']


In [11]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='ollama:llama3.2:latest')
# Generate titles and keywords for all sessions
# Display sample results
print(df_sessions_sample.head().to_string(index=False))

Using model: llama3.2:latest
Processing session 0 (1/3)...
  ✓ Generated titles for session 0
Processing session 1 (2/3)...
  ✓ Generated titles for session 1
Processing session 2 (3/3)...
  ✓ Generated titles for session 2

Total processing time: 48.33 seconds
Average time per session: 16.11 seconds
 cluster_id  session_size                                                                               gen_presentation_indices hybrid_invited_presentations final_session_title  session_coherence  session_distinctiveness                       Ollama: llama3.2:latest Title 1          Ollama: llama3.2:latest Title 2                          Ollama: llama3.2:latest Title 3                                                                                      Ollama: llama3.2:latest Keywords
          0            19 [877, 596, 778, 243, 980, 1258, 146, 1121, 1210, 1288, 439, 1110, 667, 941, 1129, 1116, 452, 133, 210]                           []         Not Set Yet           0.516857          

#### Sentence Transformers

In [ ]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='llama-3.2-local')
# Generate titles and keywords for all sessions
# df_sessions = generate_session_titles_and_keywords(df_sessions, df, topic_column)

# Display sample results
print(df_sessions_sample.head().to_string(index=False))

LLaMA model loaded successfully
Processing session 0 (1/3)...


In [ ]:
if 'model' in globals() or 'model' in locals():
    del model
    # Optionally, you can try to explicitly trigger garbage collection
    # import gc
    # gc.collect()
    print("LLaMA model has been flagged for unloading. Resources will be freed by the garbage collector.")
else:
    print("Model variable 'model' not found, or already unloaded.")

#### Gemini

In [10]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='gemini-2.0-flash')
# Generate titles and keywords for all sessions
# df_sessions = generate_session_titles_and_keywords(df_sessions, df, topic_column)

# Display sample results
print(df_sessions_sample.head().to_string(index=False))

Processing session 0 (1/3)...
  ✓ Generated titles for session 0
Processing session 1 (2/3)...
  ✓ Generated titles for session 1
Processing session 2 (3/3)...
  ✓ Generated titles for session 2

Total processing time: 7.1328 seconds
Average time per session: 2.3776 seconds
 cluster_id  session_size                                                                               gen_presentation_indices hybrid_invited_presentations final_session_title  session_coherence  session_distinctiveness                             Gemini Title 1                                  Gemini Title 2                              Gemini Title 3                                                                Gemini Keywords
          0            19 [877, 596, 778, 243, 980, 1258, 146, 1121, 1210, 1288, 439, 1110, 667, 941, 1129, 1116, 452, 133, 210]                           []         Not Set Yet           0.516857                 0.031459   AI & Robotics for Precision Weed Control      Sensing and Automat

### Match Committees to Related Sessions

In [11]:
# Read the committee file from CSV/Excel with flexible column selection
committee_file_path = 'ASABE Committees.csv'  # Update this path as needed (can also use .xlsx)

# Load committees
df_committees, committee_name_column, description_column, combined_column = session_organizer.load_committees(
    committee_file_path,
    Committee_Name_column='Committee_Name',  # Actual column name in your file
    Description_column='Description',        # Actual column name in your file
    committee_name_column='Committee_Name',  # Desired output column name
    description_column='Description',        # Desired output column name
    combined_column='Name_Description'       # Combined column for embeddings
)

print(f"Loaded {len(df_committees)} committees")
print(f"Committee name column: {committee_name_column}")
print(f"Description column: {description_column}")
print(f"Combined column: {combined_column}")
print("\nFirst few committees:")
print(df_committees[[committee_name_column, description_column]].head())

# Generate embeddings for committees using the combined column
df_committee_embeddings = session_organizer.embed_documents(df_committees, combined_column, embedding_model)

Loaded 108 committees
Committee name column: Committee_Name
Description column: Description
Combined column: Name_Description

First few committees:
                                      Committee_Name  \
0  ASE-09 Environmental Quality Coordinating Comm...   
1                          ASE-12 Forest Engineering   
2  ASE-134 Fertilizers, Soil Conditioners & US TA...   
3              ASE-16 Engineering for Sustainability   
4  ASE-347 and US TAG TC 347 Data-driven agrifood...   

                                         Description  
0  Leads and coordinates the activities of ASABE ...  
1  Forested landscapes are essential for clean wa...  
2  US Technical Advisory Group for ISO TC 134. Le...  
3  ASE-16 leads and coordinates ASABE activities ...  
4  Standardization in the field of big-picture, d...  


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [12]:
# Find the most similar committees for each session
session_committee_matches = session_organizer.find_most_similar_committees_by_presentations(
    df_sessions, 
    df_presentation_embeddings, 
    df_committees, 
    df_committee_embeddings, 
    top_n=3,
)

Processing session 0 with 19 presentations...
Processing session 1 with 13 presentations...
Processing session 2 with 15 presentations...
Processing session 3 with 14 presentations...
Processing session 4 with 18 presentations...
Processing session 5 with 14 presentations...
Processing session 6 with 11 presentations...
Processing session 7 with 16 presentations...
Processing session 8 with 15 presentations...
Processing session 9 with 27 presentations...
Processing session 10 with 11 presentations...
Processing session 11 with 11 presentations...
Processing session 12 with 14 presentations...
Processing session 13 with 26 presentations...
Processing session 14 with 13 presentations...
Processing session 15 with 16 presentations...
Processing session 16 with 13 presentations...
Processing session 17 with 19 presentations...
Processing session 18 with 16 presentations...
Processing session 19 with 12 presentations...
Processing session 20 with 15 presentations...
Processing session 21 w

In [13]:
df_sessions = session_organizer.add_committee_matches_to_clusters(df_sessions, session_committee_matches)
# Display a sample of the results
print(f"\nSample of top committee matches:")

print(df_sessions.head(10).to_string(index=False))


Sample of top committee matches:
 cluster_id  session_size                                                                                                                   gen_presentation_indices hybrid_invited_presentations final_session_title  session_coherence  session_distinctiveness                             Top Committee Match                                    2nd Committee Match                        3rd Committee Match Top Committee Similarity 2nd Committee Similarity 3rd Committee Similarity
          0            19                                     [877, 596, 778, 243, 980, 1258, 146, 1121, 1210, 1288, 439, 1110, 667, 941, 1129, 1116, 452, 133, 210]                           []         Not Set Yet           0.516857                 0.031459               MS-45 Soil-Plant-Machine Dynamics                         NRES-244 Irrigation Management   PRS-702 Crop & Feed Processing & Storage                 0.430897                 0.424146                 0.422593
        

## Hybrid Sessions Example

### Load Invited Presentaions from Hybrid Sessions

If you have existing hybrid sessions with pre-assigned presentations, you can load them.

In [5]:
# First, examine the Excel file structure
file_path = "Example Hybrid Session Invited Presentations.csv"
# Determine file type and read accordingly
if file_path.lower().endswith('.csv'):
    df_temp = pd.read_csv(file_path)
elif file_path.lower().endswith(('.xlsx', '.xls')):
    df_temp = pd.read_excel(file_path)
else:
    raise ValueError(f"Unsupported file format. Please use CSV (.csv) or Excel (.xlsx, .xls) files.")

print("Available columns:")
for i, col in enumerate(df_temp.columns):
    print(f"{i}: {col}")

print(f"\nFile contains {len(df_temp)} rows and {len(df_temp.columns)} columns")
print("\nFirst few rows preview:")
print(df_temp.head())

Available columns:
0: Session
1: Title
2: Abstract
3: Submission ID - 7 digits
4: Technical Community
5: Presenter: First Name
6: Presenter: Last Name

File contains 6 rows and 7 columns

First few rows preview:
                                             Session  \
0  AI-Powered Remote Sensing for Crop and Soil He...   
1  AI-Powered Remote Sensing for Crop and Soil He...   
2  AI-Powered Remote Sensing for Crop and Soil He...   
3  Open-Source “pyfao56” Evapotranspiration and W...   
4  Open-Source “pyfao56” Evapotranspiration and W...   

                                               Title  \
0  Weed-AI: Open and standardised sharing of anno...   
1  Unified Deep Learning Framework for Crop-Weed ...   
2  Accurate Pixel-Wise Object Detection Framework...   
3  The “pyfao56” software package for Python: Cod...   
4  The pyfao56 automatic irrigation scheduling al...   

                                            Abstract  \
0  Machine vision for weed recognition is critica...   
1 

In [6]:
# Example: Load hybrid sessions from CSV/Excel file
hybrid_file_path = "Example Hybrid Session Invited Presentations.csv"  # Update this path as needed

# Load hybrid sessions using the flexible function
df_hybrid_presentations, df_hybrid_sessions, hybrid_session_col, title_col, abstract_col, abstract_id_col, topic_col = session_organizer.load_hybrid_sessions(
    hybrid_file_path,
    Session_column='Session',              # Actual column name in your file
    Title_column='Title',                  # Actual column name in your file  
    Abstract_column='Abstract',            # Actual column name in your file
    Abstract_ID_column='Submission ID - 7 digits',  # Actual column name in your file
    session_column='Session',              # Desired output column name
    title_column='Title',                  # Desired output column name
    abstract_column='Abstract',            # Desired output column name
    abstract_id_column='Abstract ID',      # Desired output column name
    topic_column='Title and Abstract'      # Combined column for embeddings
)

print(f"Loaded {len(df_hybrid_presentations)} hybrid presentations")
print(f"Session column: {hybrid_session_col}")
print(f"Title column: {title_col}")
print(f"Abstract column: {abstract_col}")
print(f"ID column: {abstract_id_col}")
print(f"Topic column: {topic_col}")

print("\nHybrid Sessions Summary:")
print(df_hybrid_sessions[[session_organizer.COLUMNS['CLUSTER_ID'], session_organizer.COLUMNS['SESSION_SIZE'], session_organizer.COLUMNS['HYBRID_SESSION_TITLE']]].to_string(index=False))

Loaded 6 hybrid presentations in 2 sessions
Session mapping: {'AI-Powered Remote Sensing for Crop and Soil Health Monitoring': 1, 'Open-Source “pyfao56” Evapotranspiration and Water Balance Tool for Water Management': 2}
Loaded 6 hybrid presentations
Session column: Session
Title column: Title
Abstract column: Abstract
ID column: Abstract ID
Topic column: Title and Abstract

Hybrid Sessions Summary:
 cluster_id  session_size                                                                 hybrid_session_title
          1             3                        AI-Powered Remote Sensing for Crop and Soil Health Monitoring
          2             3 Open-Source “pyfao56” Evapotranspiration and Water Balance Tool for Water Management


### Embed Invited Presentations from Hybrid Sessions

In [7]:
# Generate embeddings for hybrid presentations (if needed for analysis)
df_hybrid_embeddings = session_organizer.embed_documents(df_hybrid_presentations, topic_col, embedding_model)
print(f"✓ Created embeddings for hybrid presentations with shape: {df_hybrid_embeddings.shape}")
print(f"✓ Embedding model used: {df_hybrid_embeddings[session_organizer.COLUMNS['EMBEDDING_MODEL']].iloc[0]}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Created embeddings for hybrid presentations with shape: (6, 385)
✓ Embedding model used: Unknown (sentence-transformers/all-MiniLM-L6-v2)


### Load Regular Presentaitons

In [8]:
# Load the data using the session_organizer function
file_path = "1.29.25 Abstracts.xlsx"
df, title_column, abstract_column, abstract_id_column, topic_column = session_organizer.load_presentations(
    file_path, 
    Title_name=TITLE_COLUMN,
    Abstract_name=ABSTRACT_COLUMN,
    Abstract_ID_name=ID_COLUMN
)

print(f"Loaded {len(df)} presentations successfully")
print(f"Title column: {title_column}")
print(f"Abstract column: {abstract_column}")
print(f"ID column: {abstract_id_column}")
print(f"Topic column: {topic_column}")

Loaded 1601 presentations successfully
Title column: Title
Abstract column: Abstract
ID column: Abstract ID
Topic column: Title and Abstract


### Embed Regular Presentations

In [9]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', trust_remote_code=True)
print(f"Base model: {embedding_model.model_card_data.base_model}")
df_presentation_embeddings = session_organizer.embed_documents(df, topic_column, embedding_model)
print(f"Embeddings shape: {df_presentation_embeddings.shape}")
print(f"Embedding model used: {df_presentation_embeddings[session_organizer.COLUMNS['EMBEDDING_MODEL']].iloc[0]}")

Base model: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/51 [00:00<?, ?it/s]

Embeddings shape: (1601, 385)
Embedding model used: Unknown (sentence-transformers/all-MiniLM-L6-v2)


### Remove Duplicates and Near Duplicates
This only applies to the regular presentation list.

In [10]:
similarity_threshold = 0.99
# Remove near-duplicate presentations based on the similarity threshold
df, df_presentation_embeddings = session_organizer.remove_duplicates(df, df_presentation_embeddings, similarity_func=embedding_model.similarity, threshold=similarity_threshold)

Near duplicate found: Index 41 and Index 42 (Similarity: 1.0000).
Near duplicate found: Index 77 and Index 78 (Similarity: 1.0000).
Near duplicate found: Index 77 and Index 79 (Similarity: 1.0000).
Near duplicate found: Index 78 and Index 79 (Similarity: 1.0000).
Near duplicate found: Index 156 and Index 157 (Similarity: 1.0000).
Near duplicate found: Index 221 and Index 222 (Similarity: 1.0000).
Near duplicate found: Index 223 and Index 224 (Similarity: 0.9930).
Near duplicate found: Index 269 and Index 270 (Similarity: 1.0000).
Near duplicate found: Index 279 and Index 280 (Similarity: 1.0000).
Near duplicate found: Index 325 and Index 326 (Similarity: 1.0000).
Near duplicate found: Index 354 and Index 355 (Similarity: 1.0000).
Near duplicate found: Index 410 and Index 411 (Similarity: 1.0000).
Near duplicate found: Index 458 and Index 459 (Similarity: 0.9998).
Near duplicate found: Index 476 and Index 477 (Similarity: 1.0000).
Near duplicate found: Index 496 and Index 1348 (Similari

### Create Sessions with Hybrid Sessions

In [11]:
session_column_name = 'Session Code'
df_sessions, labels, metadata = session_organizer.create_sessions_w_hybrid(df, embedding_model.similarity, df_presentation_embeddings=df_presentation_embeddings,
                                                                                     df_hybrid_presentations=df_hybrid_presentations,
                                                                                     hybrid_session_column=hybrid_session_col, df_hybrid_embeddings=df_hybrid_embeddings,
                                                                                     max_sessions=100, min_session_size=8, tree_merge_stop=1, cluster_column_name=session_column_name)
df[session_column_name] = labels
print(f"Created {metadata['n_clusters']} sessions with {metadata['n_assigned_items']} presentations.")
print(f"Unassigned Presentations: {metadata['n_unassigned_items']}")
    

Created 100 sessions with 1559 presentations.
Unassigned Presentations: 0


### Analyze Sessions

In [12]:
embeddings_only = df_presentation_embeddings.drop(columns=[session_organizer.COLUMNS['EMBEDDING_MODEL']])
pres_similarities_matrix = embedding_model.similarity(embeddings_only.values, embeddings_only.values)
# Convert to numpy if needed
if hasattr(pres_similarities_matrix, 'cpu'):
    pres_similarities_matrix = pres_similarities_matrix.cpu().numpy()
elif hasattr(pres_similarities_matrix, 'numpy'):
    pres_similarities_matrix = pres_similarities_matrix.numpy()

df['presentation_session_fit'],df_sessions['session_coherence'], df_sessions['session_distinctiveness'], df_session_session_similarity  = session_organizer.calculate_placement_metrics(
    df_presentations=df,
    df_sessions=df_sessions,
    pres_similarities_matrix=pres_similarities_matrix,
    session_column_name=session_column_name
)

### Create Session Titles & Keywords

#### Ollama

In [14]:
# Test if Ollama is accessible
import requests
try:
    response = requests.get("http://localhost:11434/api/tags")
    print(f"Ollama status: {response.status_code}")
    if response.status_code == 200:
        models = response.json()['models']
        print(f"Available models: {[m['name'] for m in models]}")
    else:
        print("Ollama server not responding correctly")
except Exception as e:
    print(f"Cannot connect to Ollama: {e}")
    print("Make sure to run 'ollama serve' first")

Ollama status: 200
Available models: ['llama3.2:latest']


In [15]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='ollama:llama3.2:latest')
# Generate titles and keywords for all sessions
# Display sample results
print(df_sessions_sample.head().to_string(index=False))

Using model: llama3.2:latest
Processing session 0 (1/3)...
  ✓ Generated titles for session 0
Processing session 1 (2/3)...
  ✓ Generated titles for session 1
Processing session 2 (3/3)...
  ✓ Generated titles for session 2

Total processing time: 31.62 seconds
Average time per session: 10.54 seconds
 cluster_id  session_size                                                 gen_presentation_indices hybrid_invited_presentations                                                                  final_session_title  session_coherence  session_distinctiveness                                                              Ollama: llama3.2:latest Title 1                                                     Ollama: llama3.2:latest Title 2                                                              Ollama: llama3.2:latest Title 3                                                                                       Ollama: llama3.2:latest Keywords
          0             7                           

#### Sentence Transformers

In [ ]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='llama-3.2-local')
# Generate titles and keywords for all sessions
# df_sessions = generate_session_titles_and_keywords(df_sessions, df, topic_column)

# Display sample results
print(df_sessions_sample.head().to_string(index=False))

In [ ]:
if 'model' in globals() or 'model' in locals():
    del model
    # Optionally, you can try to explicitly trigger garbage collection
    # import gc
    # gc.collect()
    print("LLaMA model has been flagged for unloading. Resources will be freed by the garbage collector.")
else:
    print("Model variable 'model' not found, or already unloaded.")

#### Gemini

In [13]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='gemini-2.0-flash')
# Generate titles and keywords for all sessions
# df_sessions = generate_session_titles_and_keywords(df_sessions, df, topic_column)

# Display sample results
print(df_sessions_sample.head().to_string(index=False))

Processing session 0 (1/3)...
  ✓ Generated titles for session 0
Processing session 1 (2/3)...
  ✓ Generated titles for session 1
Processing session 2 (3/3)...
  ✓ Generated titles for session 2

Total processing time: 6.9847 seconds
Average time per session: 2.3282 seconds
 cluster_id  session_size                                                 gen_presentation_indices hybrid_invited_presentations                                                                  final_session_title  session_coherence  session_distinctiveness                                             Gemini Title 1                                               Gemini Title 2                                        Gemini Title 3                                                                Gemini Keywords
          0             7                                    [777, 937, 980, 1025, 1123, 880, 619]                    [0, 1, 2]                        AI-Powered Remote Sensing for Crop and Soil Health Monitoring   

### Match Committees to Related Sessions

In [14]:
# Read the committee file from CSV/Excel with flexible column selection
committee_file_path = 'ASABE Committees.csv'  # Update this path as needed (can also use .xlsx)

# Load committees
df_committees, committee_name_column, description_column, combined_column = session_organizer.load_committees(
    committee_file_path,
    Committee_Name_column='Committee_Name',  # Actual column name in your file
    Description_column='Description',        # Actual column name in your file
    committee_name_column='Committee_Name',  # Desired output column name
    description_column='Description',        # Desired output column name
    combined_column='Name_Description'       # Combined column for embeddings
)

print(f"Loaded {len(df_committees)} committees")
print(f"Committee name column: {committee_name_column}")
print(f"Description column: {description_column}")
print(f"Combined column: {combined_column}")
print("\nFirst few committees:")
print(df_committees[[committee_name_column, description_column]].head())

# Generate embeddings for committees using the combined column
df_committee_embeddings = session_organizer.embed_documents(df_committees, combined_column, embedding_model)

Loaded 108 committees
Committee name column: Committee_Name
Description column: Description
Combined column: Name_Description

First few committees:
                                      Committee_Name  \
0  ASE-09 Environmental Quality Coordinating Comm...   
1                          ASE-12 Forest Engineering   
2  ASE-134 Fertilizers, Soil Conditioners & US TA...   
3              ASE-16 Engineering for Sustainability   
4  ASE-347 and US TAG TC 347 Data-driven agrifood...   

                                         Description  
0  Leads and coordinates the activities of ASABE ...  
1  Forested landscapes are essential for clean wa...  
2  US Technical Advisory Group for ISO TC 134. Le...  
3  ASE-16 leads and coordinates ASABE activities ...  
4  Standardization in the field of big-picture, d...  


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [15]:
# Find the most similar committees for each session
session_committee_matches = session_organizer.find_most_similar_committees_by_presentations(
    df_sessions, 
    df_presentation_embeddings, 
    df_committees, 
    df_committee_embeddings, 
    top_n=3,
)

Processing session 0 with 7 presentations...
Processing session 1 with 9 presentations...
Processing session 2 with 13 presentations...
Processing session 3 with 16 presentations...
Processing session 4 with 14 presentations...
Processing session 5 with 22 presentations...
Processing session 6 with 16 presentations...
Processing session 7 with 18 presentations...
Processing session 8 with 11 presentations...
Processing session 9 with 14 presentations...
Processing session 10 with 16 presentations...
Processing session 11 with 29 presentations...
Processing session 12 with 12 presentations...
Processing session 13 with 15 presentations...
Processing session 14 with 23 presentations...
Processing session 15 with 17 presentations...
Processing session 16 with 12 presentations...
Processing session 17 with 20 presentations...
Processing session 18 with 16 presentations...
Processing session 19 with 12 presentations...
Processing session 20 with 15 presentations...
Processing session 21 wit

In [16]:
df_sessions = session_organizer.add_committee_matches_to_clusters(df_sessions, session_committee_matches)
# Display a sample of the results
print(f"\nSample of top committee matches:")

print(df_sessions.head(10).to_string(index=False))


Sample of top committee matches:
 cluster_id  session_size                                                                                            gen_presentation_indices hybrid_invited_presentations                                                                  final_session_title  session_coherence  session_distinctiveness                             Top Committee Match                                    2nd Committee Match                                    3rd Committee Match Top Committee Similarity 2nd Committee Similarity 3rd Committee Similarity
          0             7                                                                               [777, 937, 980, 1025, 1123, 880, 619]                    [0, 1, 2]                        AI-Powered Remote Sensing for Crop and Soil Health Monitoring           0.582196                 0.129191            NRES-246 Turf & Landscape Irrigation                         NRES-244 Irrigation Management                          MS-60

## Analyze Manually Edited Session Placements
After session creation, organizers will change placements. This code loads those placements and calculates the similarities of these sessions.

Presentations are placed into sessions based on them having the same session code in the session code column of the presentation spreadsheet. A session dataframe is then created to match the presentation dataframe.

### Check Data File Format

In [5]:
# First, examine the Excel file structure
file_path = "Presentations - Manual Placement.csv"
df_temp = pd.read_csv(file_path)

print("Available columns:")
for i, col in enumerate(df_temp.columns):
    print(f"{i}: {col}")

print(f"\nFile contains {len(df_temp)} rows and {len(df_temp.columns)} columns")
print("\nFirst few rows preview:")
print(df_temp.head())

Available columns:
0: Session
1: Title
2: Abstract
3: Abstract ID
4: Technical Community
5: Profile: First Name
6: Profile: Last Name
7: Title and Abstract
8: Session Code
9: presentation_session_fit

File contains 1559 rows and 10 columns

First few rows preview:
                                             Session  \
0  Thermochemical and Catalytic Conversion of Bio...   
1  Value-Added Chemicals Products and Materials t...   
2  Innovations in Precision Agriculture and Smart...   
3  Conservation Drainage Practices – Current and ...   
4  Water Management and Soil Health under Water S...   

                                               Title  \
0  Catalytic hydrothermal gasification of pinecon...   
1  Progressive Closed-Loop Technologies from Biom...   
2  High Clearance Robotic Irrigation Impacts on S...   
3  Accelerating the adoption of saturated buffers...   
4  Evaluation of Crop Water Use and Productivity ...   

                                            Abstract  Abstrac

In [6]:
# Define your column selections based on the output above
TITLE_COLUMN = 'Title'  # Update based on your file
ABSTRACT_COLUMN = 'Abstract'  # Update based on your file  
ID_COLUMN = 'Abstract ID'  # Update based on your file
SESSION = 'Session Code'  # Update based on your file

### Load Manually Placed Presentations

In [7]:
# Load the data using the session_organizer function
df, title_column, abstract_column, abstract_id_column, topic_column = session_organizer.load_presentations(
    file_path, 
    Title_name=TITLE_COLUMN,
    Abstract_name=ABSTRACT_COLUMN,
    Abstract_ID_name=ID_COLUMN
)

print(f"Loaded {len(df)} presentations successfully")
print(f"Title column: {title_column}")
print(f"Abstract column: {abstract_column}")
print(f"ID column: {abstract_id_column}")
print(f"Topic column: {topic_column}")

Loaded 1559 presentations successfully
Title column: Title
Abstract column: Abstract
ID column: Abstract ID
Topic column: Title and Abstract


### Perform the Embedding

In [8]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', trust_remote_code=True)
print(f"Base model: {embedding_model.model_card_data.base_model}")
df_presentation_embeddings = session_organizer.embed_documents(df, topic_column, embedding_model)
print(f"Embeddings shape: {df_presentation_embeddings.shape}")
print(f"Embedding model used: {df_presentation_embeddings[session_organizer.COLUMNS['EMBEDDING_MODEL']].iloc[0]}")

Base model: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/49 [00:00<?, ?it/s]

Embeddings shape: (1559, 385)
Embedding model used: Unknown (sentence-transformers/all-MiniLM-L6-v2)


### Remove Duplicates and Near-Duplicates

In [9]:
similarity_threshold = 0.99
# Remove near-duplicate presentations based on the similarity threshold
df, df_presentation_embeddings = session_organizer.remove_duplicates(df, df_presentation_embeddings, similarity_func=embedding_model.similarity, threshold=similarity_threshold)


Found 0 near-duplicate presentations to remove (keeping highest index).
Indices to remove: []

Final number of oral presentations: 1559
Final shape of embeddings matrix: (1559, 385)


### Create Sessions

In [ ]:
def create_sessions_from_assignments(df_edited, session_column='Session Code'):
    """Create df_sessions from manually assigned session codes"""

    # Get unique session codes (excluding -1 for unassigned)
    assigned_df = df_edited[df_edited[session_column] != -1]
    
    if assigned_df.empty:
        # No sessions assigned
        df_sessions = pd.DataFrame()
        labels = pd.Series(-1, index=df_edited.index, name=session_column)
        metadata = {
            'n_clusters': 0,
            'n_assigned_items': 0,
            'n_unassigned_items': len(df_edited),
            'n_total_items': len(df_edited),
            'source': 'manually_edited'
        }
        return
    print("Creating sessions from manually assigned session codes...")
    # Group by session code to create clusters
    session_groups = assigned_df.groupby(session_column)
    final_clusters_df_indices = []
    
    for session_code, group in session_groups:
        # Get the DataFrame indices for this session
        cluster_indices = group.index.tolist()
        final_clusters_df_indices.append(cluster_indices)
    
    # Sort clusters by session code for consistency
    final_clusters_df_indices.sort(key=lambda cluster: df_edited.loc[cluster[0], session_column])
    
    # Prepare hybrid data - preserve existing hybrid sessions in their exact locations
    hybrid_cluster_presentations = []
    hybrid_session_titles = []
    
    
    # Ignore hybrid data, use defaults for all clusters
    hybrid_cluster_presentations = [[] for _ in final_clusters_df_indices]
    hybrid_session_titles = [session_organizer.UNSET_SESSION_TITLE_TEXT for _ in final_clusters_df_indices]
    
    # Create output structures using the existing function
    return session_organizer._create_output_structures_with_df_indices(
        final_clusters_df_indices, 
        df_edited, 
        session_column,
        hybrid_cluster_presentations, 
        hybrid_session_titles, 
    )
    

In [ ]:
session_column_name = 'Session Code'
# The create_sessions_from_assignments function returns labels that are -1 for unassigned presentations.
# It also ensures that sessions codes are sequential.
# This is important to maintain consistency with the session codes in df_sessions.
# These labels will match the session codes in df_sessions.
df_sessions, labels, metadata = create_sessions_from_assignments(df)
# Update the original DataFrame with new session label codes. 
df[session_column_name] = labels
print(f"Created {metadata['n_clusters']} sessions with {metadata['n_assigned_items']} presentations.")
print(f"Unassigned Presentations: {metadata['n_unassigned_items']}")

Creating sessions from manually assigned session codes...
Created 101 sessions with 1558 presentations.
Unassigned Presentations: 1


### Analyze Sessions

- session_coherence = "Are presentations within this session similar?" (internal session quality)
- session_distinctiveness = "Is this session's topic unique compared to others?" (relative session positioning)
- presentation_session_fit = "Does this presentation match the topic of others in the session?" (presentation fit)

Session Coherence measures cluster cohesion. It reflects how tighly grouped the topic of presentations within the session are.

Session Distinctiveness measures how unique each session's topic is. High values mean the session has a clear, focused theme that's different from other sessions. Low values suggest either the session mixes different topics or overlaps too much with other sessions.

Presentation-Session Fit is an individual presentations's average similarity to other presentation in its session. Generically, it can be referred to as "within_cluster_fit", "cluster_membership_strength", or "local_cohesion_score".

In [15]:
embeddings_only = df_presentation_embeddings.drop(columns=[session_organizer.COLUMNS['EMBEDDING_MODEL']])
pres_similarities_matrix = embedding_model.similarity(embeddings_only.values, embeddings_only.values)
# Convert to numpy if needed
if hasattr(pres_similarities_matrix, 'cpu'):
    pres_similarities_matrix = pres_similarities_matrix.cpu().numpy()
elif hasattr(pres_similarities_matrix, 'numpy'):
    pres_similarities_matrix = pres_similarities_matrix.numpy()

df['presentation_session_fit'],df_sessions['session_coherence'], df_sessions['session_distinctiveness'], df_session_session_similarity  = session_organizer.calculate_placement_metrics(
    df_presentations=df,
    df_sessions=df_sessions,
    pres_similarities_matrix=pres_similarities_matrix,
    session_column_name=session_column_name
)

### Create Session Titles & Keywords

#### Ollama

In [ ]:
# Test if Ollama is accessible
import requests
try:
    response = requests.get("http://localhost:11434/api/tags")
    print(f"Ollama status: {response.status_code}")
    if response.status_code == 200:
        models = response.json()['models']
        print(f"Available models: {[m['name'] for m in models]}")
    else:
        print("Ollama server not responding correctly")
except Exception as e:
    print(f"Cannot connect to Ollama: {e}")
    print("Make sure to run 'ollama serve' first")

In [ ]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='ollama:llama3.2:latest')
# Generate titles and keywords for all sessions
# Display sample results
print(df_sessions_sample.head().to_string(index=False))

#### Sentence Transformers

In [ ]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='llama-3.2-local')
# Generate titles and keywords for all sessions
# df_sessions = generate_session_titles_and_keywords(df_sessions, df, topic_column)

# Display sample results
print(df_sessions_sample.head().to_string(index=False))

In [ ]:
if 'model' in globals() or 'model' in locals():
    del model
    # Optionally, you can try to explicitly trigger garbage collection
    # import gc
    # gc.collect()
    print("LLaMA model has been flagged for unloading. Resources will be freed by the garbage collector.")
else:
    print("Model variable 'model' not found, or already unloaded.")

#### Gemini

In [ ]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='gemini-2.0-flash')
# Generate titles and keywords for all sessions
# df_sessions = generate_session_titles_and_keywords(df_sessions, df, topic_column)

# Display sample results
print(df_sessions_sample.head().to_string(index=False))

### Match Committees to Related Sessions

In [ ]:
# Read the committee file from CSV/Excel with flexible column selection
committee_file_path = 'ASABE Committees.csv'  # Update this path as needed (can also use .xlsx)

# Load committees
df_committees, committee_name_column, description_column, combined_column = session_organizer.load_committees(
    committee_file_path,
    Committee_Name_column='Committee_Name',  # Actual column name in your file
    Description_column='Description',        # Actual column name in your file
    committee_name_column='Committee_Name',  # Desired output column name
    description_column='Description',        # Desired output column name
    combined_column='Name_Description'       # Combined column for embeddings
)

print(f"Loaded {len(df_committees)} committees")
print(f"Committee name column: {committee_name_column}")
print(f"Description column: {description_column}")
print(f"Combined column: {combined_column}")
print("\nFirst few committees:")
print(df_committees[[committee_name_column, description_column]].head())

# Generate embeddings for committees using the combined column
df_committee_embeddings = session_organizer.embed_documents(df_committees, combined_column, embedding_model)

In [ ]:
# Find the most similar committees for each session
session_committee_matches = session_organizer.find_most_similar_committees_by_presentations(
    df_sessions, 
    df_presentation_embeddings, 
    df_committees, 
    df_committee_embeddings, 
    top_n=3,
)

In [ ]:
df_sessions = session_organizer.add_committee_matches_to_clusters(df_sessions, session_committee_matches)
# Display a sample of the results
print(f"\nSample of top committee matches:")

print(df_sessions.head(10).to_string(index=False))

## Verification
Test to make sure all presentations are included.

In [ ]:
from itertools import chain

all_indices = list(chain.from_iterable(df_sessions[session_organizer.COLUMNS['GEN_PRESENTATION_INDICES']]))
max_index = max(all_indices)
min_index = min(all_indices)
unique_indices = set(all_indices)
expected_indices = set(range(min_index, max_index + 1))
missing_indices = sorted(expected_indices - unique_indices)

print(f"Max index: {max_index}")
print(f"Min index: {min_index}")
print(f"Number of unique indices: {len(unique_indices)}")
print(f"Missing indices: {missing_indices}")
print(f"Any indices skipped? {'Yes' if missing_indices else 'No'}")